# 📊 Customer Churn Risk & RFM Segmentation Analysis

**Dataset:** Online Retail Transactions  
**Technique:** RFM (Recency · Frequency · Monetary) Analysis  
**Tools:** Python · Pandas · Matplotlib · Seaborn  
**Author:** Anurag Kumar Singh

---

## 🎯 Objective
Segment customers based on their purchasing behavior using RFM scores to:
- Identify high-value **Champions** driving the most revenue
- Flag **At Risk** customers for targeted retention campaigns
- Enable data-driven, segment-specific marketing strategies

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
sns.set_style("whitegrid")
sns.set_palette("muted")
plt.rcParams["figure.dpi"] = 100

print("✅ Libraries loaded successfully")

---
## 2. Load & Preview Data

In [ ]:
# Load dataset
df = pd.read_excel("Online_Retail.xlsx")

print(f"Dataset shape: {df.shape}")
df.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset info and missing values
print("=== Dataset Info ===")
df.info()

print("\n=== Missing Values ===")
print(df.isnull().sum())

In [ ]:
# Statistical summary
print("=== Statistical Summary ===")
df.describe()

> **Key Observations:**
> - `CustomerID` has ~135,080 missing values — these transactions cannot be tied to a customer, so they'll be removed.
> - `Quantity` and `UnitPrice` contain **negative values** (returns/cancellations) — these will be filtered out.
> - Data spans from **Dec 2010 to Dec 2011**.

---
## 4. Data Cleaning & Preprocessing

In [ ]:
# Step 1: Remove rows with missing CustomerID
df = df[df["CustomerID"].notna()]

# Step 2: Remove returns and cancellations (negative Quantity or UnitPrice)
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# Step 3: Ensure InvoiceDate is in datetime format
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Step 4: Create Revenue column
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print(f"✅ Cleaned dataset shape: {df.shape}")
df.head()

---
## 5. RFM Feature Engineering

RFM scores are calculated based on:
- **Recency (R):** Days since last purchase (lower = better)
- **Frequency (F):** Number of unique invoices/orders
- **Monetary (M):** Total revenue generated

In [ ]:
# Snapshot date = 1 day after the last transaction
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")

# Aggregate RFM metrics per customer
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

print(f"\n✅ RFM table shape: {rfm.shape}")
rfm.head()

---
## 6. RFM Scoring (Quintile-Based)

Each RFM metric is divided into 5 quintiles (1–5):
- **R Score:** 5 = most recent (low recency value)
- **F Score:** 5 = most frequent
- **M Score:** 5 = highest spender

In [ ]:
# R Score: lower recency → higher score (inverted scale)
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1])

# F Score: higher frequency → higher score
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

# M Score: higher monetary → higher score
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5])

# Convert scores to integer
rfm[["R_Score", "F_Score", "M_Score"]] = rfm[["R_Score", "F_Score", "M_Score"]].astype(int)

# Combine into a single RFM composite score (string)
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm.head()

---
## 7. Customer Segmentation

Customers are classified into 6 segments based on their RFM scores:

In [ ]:
def segment_customer(row):
    """Classify customers into segments based on RFM scores."""
    if row["RFM_Score"] == "555":
        return "Champions"
    elif row["R_Score"] >= 4 and row["F_Score"] >= 4:
        return "Loyal Customers"
    elif row["R_Score"] >= 4:
        return "Recent Customers"
    elif row["F_Score"] >= 4:
        return "Frequent Customers"
    elif row["M_Score"] >= 4:
        return "Big Spenders"
    else:
        return "At Risk"


rfm["Segment"] = rfm.apply(segment_customer, axis=1)

print("=== Segment Distribution ===")
print(rfm["Segment"].value_counts())

---
## 8. Business KPIs

In [ ]:
total_customers   = rfm.shape[0]
total_revenue     = rfm["Monetary"].sum()
avg_customer_value = rfm["Monetary"].mean()
avg_frequency     = rfm["Frequency"].mean()
avg_recency       = rfm["Recency"].mean()
revenue_per_customer = total_revenue / total_customers

kpi = pd.DataFrame({
    "Metric": [
        "Total Active Customers",
        "Total Revenue",
        "Avg Customer Spend",
        "Avg Purchase Frequency",
        "Avg Recency (days)",
        "Revenue per Customer",
    ],
    "Value": [
        f"{total_customers:,}",
        f"£{total_revenue:,.2f}",
        f"£{avg_customer_value:,.2f}",
        f"{avg_frequency:.2f}",
        f"{avg_recency:.2f}",
        f"£{revenue_per_customer:,.2f}",
    ]
})

print("=== Key Performance Indicators ===")
kpi

In [ ]:
# Revenue contribution by segment
print("=== Revenue by Segment ===")
rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False).apply(lambda x: f"£{x:,.2f}")

---
## 9. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Chart 1: Customer Count by Segment ---
seg_counts = rfm["Segment"].value_counts()
axes[0].bar(seg_counts.index, seg_counts.values, color=sns.color_palette("muted", len(seg_counts)))
axes[0].set_title("Customer Segment Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Number of Customers")
axes[0].set_xlabel("Segment")
axes[0].tick_params(axis="x", rotation=30)

# --- Chart 2: Revenue by Segment ---
seg_revenue = rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)
axes[1].bar(seg_revenue.index, seg_revenue.values, color=sns.color_palette("coolwarm", len(seg_revenue)))
axes[1].set_title("Revenue Contribution by Segment", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Total Revenue (£)")
axes[1].set_xlabel("Segment")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("segment_overview.png", bbox_inches="tight")
plt.show()

In [ ]:
# --- RFM Heatmap: R vs F Score with Avg Monetary Value ---
pivot = rfm.pivot_table(
    index="R_Score",
    columns="F_Score",
    values="Monetary",
    aggfunc="mean"
)

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, cmap="coolwarm", annot=True, fmt=".0f", linewidths=0.5)
plt.title("Avg Monetary Value: Recency vs Frequency Scores", fontsize=13, fontweight="bold")
plt.xlabel("Frequency Score")
plt.ylabel("Recency Score")
plt.tight_layout()
plt.savefig("rfm_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# --- RFM Correlation Heatmap ---
plt.figure(figsize=(6, 5))
sns.heatmap(
    rfm[["Recency", "Frequency", "Monetary"]].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)
plt.title("RFM Feature Correlation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("rfm_correlation.png", bbox_inches="tight")
plt.show()

In [ ]:
# --- Top 10 High-Value Customers ---
top_customers = rfm.sort_values("Monetary", ascending=False).head(10)

plt.figure(figsize=(10, 5))
plt.bar(
    top_customers["CustomerID"].astype(str),
    top_customers["Monetary"],
    color=sns.color_palette("Blues_d", 10)
)
plt.title("Top 10 High-Value Customers by Monetary Score", fontsize=13, fontweight="bold")
plt.xlabel("Customer ID")
plt.ylabel("Total Spend (£)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("top_customers.png", bbox_inches="tight")
plt.show()

---
## 10. Business Insights

| Segment | Finding | Recommended Action |
|---|---|---|
| 🏆 **Champions** | Highest revenue drivers — small group, massive impact | Reward loyalty; early access to new products |
| 💙 **Loyal Customers** | Purchase frequently but lower spend per order | Upsell / cross-sell to increase basket size |
| ⚠️ **At Risk** | Highest volume segment — need re-engagement | Launch retention campaigns, personalized offers |
| 💰 **Big Spenders** | High-value but low frequency | Win-back campaigns to encourage repeat purchase |
| 🔁 **Frequent Customers** | Regular buyers, moderate spend | Incentivize spending via bundles / loyalty tiers |
| 🆕 **Recent Customers** | Newly acquired — retention is critical | Onboarding sequence, welcome offers |

---
## 11. Export Results

In [ ]:
# Full RFM segmentation output
rfm.to_csv("Customer_RFM_Segmentation.csv", index=False)
print("✅ Full RFM data saved → Customer_RFM_Segmentation.csv")

# At-risk customers for retention campaigns
at_risk = rfm[rfm["Segment"] == "At Risk"]
at_risk.to_csv("At_Risk_Customers.csv", index=False)
print(f"✅ At-risk customers saved → At_Risk_Customers.csv ({len(at_risk):,} customers)")

# Top 10 high-value customers
top_customers.to_csv("Top_10_Customers.csv", index=False)
print("✅ Top 10 customers saved → Top_10_Customers.csv")